### Exception order

In [1]:
class IncorrectDataError(ValueError):  # base exception
    """Exception, that will be raised, if incorrect data is passed"""
    
class EmptyNameError(IncorrectDataError):
    """Exception if name is an empty string."""
    
class NegativeAgeError(IncorrectDataError):
    """Exception if age < 0."""
    
class NicknameNotProvidedError(IncorrectDataError):
    """Exception if 'nickname' is not provided."""

def user_info(user):
    if len(user["name"]) == 0:
        raise EmptyNameError("Name cannot be empty.")
        
    if user["age"] < 0:
        raise NegativeAgeError("Age must be > 0")
        
    if "nickname" not in user:
        raise NicknameNotProvidedError("Key 'nickname' must be provided.")
        
    return f"{user['nickname']} - {user['name']} (age: {user['age']})"

In [5]:
try:
    print(user_info({
        "name": "John",
        "age": 25,
#         "nickname": "johnny-005"
    }))
except IncorrectDataError as e:
    print(f"Please, fix your data. Error: {e}")

Please, fix your data. Error: Key 'nickname' must be provided.


In [6]:
try:
    print(user_info({
        "name": "John",
        "age": 25,
#         "nickname": "johnny-005"
    }))
except IncorrectDataError as e:
    print(f"Please, fix your data. Error: {e}")
except NicknameNotProvidedError:
    print("You must select nickname first.")

Please, fix your data. Error: Key 'nickname' must be provided.


because of hierarchy of our Exception classes we built we still cannot catch the `NicknameNotProvidedError` as it inherited from `IncorrectDataError` and it's shadowed now

In [7]:
try:
    print(user_info({
        "name": "John",
        "age": 25,
#         "nickname": "johnny-005"
    }))
except NicknameNotProvidedError:
    print("You must select nickname first.")
except IncorrectDataError as e:
    print(f"Please, fix your data. Error: {e}")

You must select nickname first.


after we have swapped the order of `except` statement we got right result we wanted to

In [8]:
try:
    print(user_info({
        "name": "John",
        "age": -5,
        "nickname": "johnny-005"
    }))
except NicknameNotProvidedError:
    print("You must select nickname first.")
except IncorrectDataError as e:
    print(f"Please, fix your data. Error: {e}")

Please, fix your data. Error: Age must be > 0


the most right order should be:

In [12]:
try:
    print(user_info({
        "name": "",
        "age": 25,
        "nickname": "johnny-005"
    }))
except NegativeAgeError:
    print("Here will be caught age error")
except NicknameNotProvidedError:
    print("Here - nickname error")
except IncorrectDataError:
    print("Here only empty name error")
except Exception as e:
    print("Here - all other kinds of errors")
    print(e)

Here only empty name error


**The most general Exception type must be caught at a most latter stage**.

### Re-raise Exception

In [2]:
numbers_to_divide = [
    (15, 5),
    (16, 8),
    (10, 0),
    (5, 5)
]

In [3]:
def divide(a, b):
    try:
        return a // b
    except ZeroDivisionError:
        print(f"Ooops. Unable to divide {a} // {b} :<")

In [16]:
result = []
for numerator, denomirator in numbers_to_divide:
    try:
        result.append(divide(numerator, denomirator))
    except ZeroDivisionError:
        result.append(0)
        
print(result)

Ooops. Unable to divide 10 // 0 :<
[3, 2, None, 1]


instead of appending `0` to `result` when `ZeroDivisionError` occured we got `None` in there

the point is, that `ZeroDivisionError` is caught in the `divide()`, and there is implicitly returned `None` which is then appended to the `result`

the solution is to ***re-raise*** exception:

In [4]:
def divide(a, b):
    try:
        return a // b
    except ZeroDivisionError:
        print(f"Ooops. Unable to divide {a} // {b} :<")
        raise  # just use 'raise' keyword, and it will re-raise 'ZeroDivisionError'

In [5]:
result = []
for numerator, denomirator in numbers_to_divide:
    try:
        result.append(divide(numerator, denomirator))
    except ZeroDivisionError:
        result.append(0)
        
print(result)

Ooops. Unable to divide 10 // 0 :<
[3, 2, 0, 1]


after ***exception re-raise*** in `divide()` we get it correctly in our for loop `try...except` block and handle it as excpected

In [6]:
def divide(a, b):
    try:
        return a // b
    except ZeroDivisionError:
        print(f"Ooops. Unable to divide {a} // {b} :<")
        raise
        
result = []
for numerator, denomirator in numbers_to_divide:
    try:
        result.append(divide(numerator, denomirator))
    except ZeroDivisionError:
        result.append(0)
        raise  # re-raise 'ZeroDivisionError' again here, and we get it with traceback
        
print(result)

Ooops. Unable to divide 10 // 0 :<


ZeroDivisionError: integer division or modulo by zero